In [1]:
import tensorflow as tf

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'sklearn'

In [2]:
train_path = r"E:\study\4th year\8th\thesis\datasets\alzheimers multiclass\train"
test_path = r"E:\study\4th year\8th\thesis\datasets\alzheimers multiclass\test"

# test_path2 = r"E:\study\4th year\8th\thesis\datasets\alzheimers detection\Data"

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    shuffle=False,
    image_size=(224,224),
    batch_size=32
)
# test_ds2 = tf.keras.utils.image_dataset_from_directory(
#     test_path2,
#     shuffle=False,
#     image_size=(224,224),
#     batch_size=32
# )

Found 55443 files belonging to 4 classes.
Using 44355 files for training.
Found 55443 files belonging to 4 classes.
Using 11088 files for validation.
Found 46184 files belonging to 4 classes.


In [3]:
print(train_ds.class_names)

['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']


In [4]:
# model = models.Sequential([
#     layers.Input(shape=(224, 224, 3)),
#     layers.Rescaling(1./255),
#     layers.Conv2D(32, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
#     layers.Conv2D(64, (3, 3), activation='relu'),
    
#     # layers.Flatten(),
#     layers.GlobalAveragePooling2D()
#     layers.Dense(64, activation='relu'),
#     layers.Dense(4)
#     ])

In [5]:
# model.compile(optimizer='adam',
#               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
#               metrics=['accuracy'])


# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=10
# )

In [6]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False # Freeze the base initially

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,


    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(4)
    
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 15s 2us/step



In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 671s 478ms/step - accuracy: 0.6925 - loss: 0.7113 - val_accuracy: 0.7835 - val_loss: 0.4905
Epoch 2/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 665s 479ms/step - accuracy: 0.7542 - loss: 0.5573 - val_accuracy: 0.8037 - val_loss: 0.4453
Epoch 3/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 687s 495ms/step - accuracy: 0.7754 - loss: 0.5147 - val_accuracy: 0.8239 - val_loss: 0.4153
Epoch 4/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 656s 473ms/step - accuracy: 0.7936 - loss: 0.4811 - val_accuracy: 0.8414 - val_loss: 0.3817
Epoch 5/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 652s 470ms/step - accuracy: 0.8048 - loss: 0.4551 - val_accuracy: 0.8415 - val_loss: 0.3653
Epoch 6/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 679s 489ms/step - accuracy: 0.8143 - loss: 0.4364 - val_accuracy: 0.8540 - val_loss: 0.3467
Epoch 7/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 680s 491ms/step - accuracy: 0.8266 - loss: 0.4113 - val_accuracy: 0.8647 - val_loss: 0.3335
Epoch 8/10
1387/1387 ━━━━━━━━━━━━━━━━━━━━ 649s 468ms/step - ac

In [8]:
model.evaluate(test_ds)

1444/1444 ━━━━━━━━━━━━━━━━━━━━ 529s 366ms/step - accuracy: 0.6626 - loss: 1.0857


[1.0856631994247437, 0.6626104116439819]

In [10]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,761,678 (10.53 MB)

 Trainable params: 167,044 (652.52 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

 Optimizer params: 334,090 (1.27 MB)

In [11]:
history.history

{'accuracy': [0.6925487518310547,
  0.7542328834533691,
  0.7754255533218384,
  0.7935971021652222,
  0.804847240447998,
  0.8142712116241455,
  0.8265810012817383,
  0.8273475170135498,
  0.838823139667511,
  0.8428813219070435],
 'loss': [0.7112749218940735,
  0.557338535785675,
  0.5147422552108765,
  0.48110565543174744,
  0.4550631046295166,
  0.4364452064037323,
  0.4112829566001892,
  0.4046052396297455,
  0.388572096824646,
  0.37740522623062134],
 'val_accuracy': [0.7835497856140137,
  0.8036616444587708,
  0.8238636255264282,
  0.8413600325584412,
  0.8415403962135315,
  0.8539862632751465,
  0.8647186160087585,
  0.868686854839325,
  0.8655303120613098,
  0.8736472129821777],
 'val_loss': [0.4904547333717346,
  0.4453125,
  0.415302574634552,
  0.3816651701927185,
  0.36533522605895996,
  0.34674251079559326,
  0.33347088098526,
  0.320436030626297,
  0.32657358050346375,
  0.30392536520957947]}